# 02 — Modelagem

Esse notebook aborda:

> TODO (Gabriel): listar as etapas do checklist cobertas aqui.

## Setup

In [77]:
from pathlib import Path

import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

SEED = 42
DATA_DIR = Path("../data")

## 4. Preparar os dados para os algoritmos

> TODO (Gabriel).

In [76]:
df_cardio = pd.read_csv(DATA_DIR / "cardio_train.csv", index_col=0)
df_cardio["age_year"] = (df_cardio["age"]/365.25)
df_cardio = df_cardio.drop(columns=["id","age"])

df_cardio_unique = df_cardio.drop_duplicates()
df_cardio_corrigido = df_cardio_unique.copy()

negativos = (df_cardio_corrigido["ap_hi"] < 0) | (df_cardio_corrigido["ap_lo"] < 0)
df_cardio_corrigido.loc[negativos, ["ap_hi", "ap_lo"]] = df_cardio_corrigido.loc[negativos, ["ap_hi", "ap_lo"]].abs() 
ap_hi_abaixo = df_cardio_corrigido["ap_hi"].between(1,29)
df_cardio_corrigido.loc[ap_hi_abaixo, "ap_hi"] = df_cardio_corrigido.loc[ap_hi_abaixo, "ap_hi"]*10

ap_lo_acima = df_cardio_corrigido["ap_lo"].between(301, 3000)
df_cardio_corrigido.loc[ap_lo_acima, "ap_lo"] = df_cardio_corrigido.loc[ap_lo_acima, "ap_lo"]//10

trocadas = df_cardio_corrigido["ap_lo"] > df_cardio_corrigido["ap_hi"]
df_cardio_corrigido.loc[trocadas, ["ap_lo", "ap_hi"]] = df_cardio_corrigido.loc[trocadas, ["ap_hi","ap_lo"]].values

limites_ap_hi = ~df_cardio_corrigido["ap_hi"].between(60,250)
df_cardio_corrigido = df_cardio_corrigido[~limites_ap_hi]

limites_ap_lo = ~df_cardio_corrigido["ap_lo"].between(30,200)
df_cardio_corrigido = df_cardio_corrigido[~limites_ap_lo] 

limites_height = ~df_cardio_corrigido["height"].between(120,220)
df_cardio_corrigido = df_cardio_corrigido[~limites_height]

limites_weight = ~df_cardio_corrigido["weight"].between(30,200)
df_cardio_corrigido = df_cardio_corrigido[~limites_weight]

regras = {
    "pressão negativa ou zero": (df_cardio_corrigido["ap_hi"] <= 0) | (df_cardio_corrigido["ap_lo"] <= 0),
    "ap_hi fora de 60–250": ~df_cardio_corrigido["ap_hi"].between(60, 250),
    "ap_lo fora de 30–200": ~df_cardio_corrigido["ap_lo"].between(30, 200),
    "ap_lo maior que ap_hi": df_cardio_corrigido["ap_lo"] > df_cardio_corrigido["ap_hi"],
    "altura fora de 120–220 cm": ~df_cardio_corrigido["height"].between(120, 220),
    "peso fora de 30–200 kg": ~df_cardio_corrigido["weight"].between(30, 200),
}

display(pd.DataFrame({
    "Linhas": {nome: mascara.sum() for nome, mascara in regras.items()},
    "% do dataset": {nome: round(mascara.mean() * 100, 2) for nome, mascara in regras.items()},
}))

outliers_irreais = pd.concat(regras, axis=1).any(axis=1)
display(f"Linhas que alguma regra descartaria: {outliers_irreais.sum()} ({outliers_irreais.mean():.2%})")


colunas_numericas = ["age_year", "height", "weight", "ap_lo", "ap_hi"]
fig = make_subplots(rows=len(colunas_numericas), cols= 1)
cnt = 1

for col in colunas_numericas:
    fig.add_trace(go.Box(x=df_cardio_corrigido[col], name=col), row=cnt, col=1)
    cnt+=1

fig.show()

df_cardio_corrigido.shape




,Linhas,% do dataset
pressão negativa ou zero,0,0.0
ap_hi fora de 60–250,0,0.0
ap_lo fora de 30–200,0,0.0
ap_lo maior que ap_hi,0,0.0
altura fora de 120–220 cm,0,0.0
peso fora de 30–200 kg,0,0.0


'Linhas que alguma regra descartaria: 0 (0.00%)'

(69806, 12)

## 5. Selecionar modelo e treiná-lo

In [125]:
X = df_cardio_corrigido.drop(columns="cardio")
Y = df_cardio_corrigido["cardio"]
x_train_clean, x_test_clean, y_train_clean, y_test_clean = train_test_split(X, Y, test_size=0.3, random_state=SEED, stratify=Y)



Uma categoria de cada (Géron 5.1): linear, Naive Bayes, SVM, vizinhos, árvore, *ensemble* e rede neural, mais o **chute** como referência mínima (Géron 1.7). Todos usam parâmetros padrão; só foram acrescentados o `random_state=SEED` (reprodutibilidade, sklearn 12.3) e o `StandardScaler` dentro do `Pipeline` para os modelos sensíveis à escala (3.9, item 14).

Métricas da 3.9 (item 16): **recall** é a prioridade, **precisão** impede o modelo "todos doentes", e a **ROC AUC** resume as duas em todos os limiares. O desvio entre os 5 folds mostra se uma diferença é maior que o ruído (Géron 5.2).

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

modelos = {
    "Chute (Dummy)": DummyClassifier(strategy="stratified", random_state=SEED),
    "Regressão Logística": make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED)),
    "Naive Bayes": GaussianNB(),
    "SVM linear": make_pipeline(StandardScaler(), LinearSVC(random_state=SEED)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED, n_jobs=4),
    "Rede Neural (MLP)": make_pipeline(StandardScaler(), MLPClassifier(random_state=SEED)),
}

resultados = []
for nome, modelo in modelos.items():
    cv = cross_validate(modelo, x_train_clean, y_train_clean, cv=5, scoring=["precision", "recall", "roc_auc"])
    resultados.append({
        "Modelo": nome,
        "Precisão": cv["test_precision"].mean(),
        "Recall": cv["test_recall"].mean(),
        "Recall (desvio)": cv["test_recall"].std(),
        "ROC AUC": cv["test_roc_auc"].mean(),
        "ROC AUC (desvio)": cv["test_roc_auc"].std(),
    })

display(pd.DataFrame(resultados).round(4))

,Modelo,Precisão,Recall,Recall (desvio),ROC AUC,ROC AUC (desvio)
0,Chute (Dummy),0.5027,0.4943,0.0056,0.5029,0.0056
1,Regressão Logística,0.7562,0.6735,0.0101,0.7918,0.0077
2,Naive Bayes,0.7647,0.6085,0.0052,0.7787,0.0089
3,SVM linear,0.7590,0.6650,0.0083,0.7916,0.0077
4,KNN,0.6975,0.6836,0.0059,0.7419,0.0062
5,Árvore de Decisão,0.6315,0.6330,0.0043,0.6320,0.0016
6,Random Forest,0.7188,0.6998,0.0041,0.7736,0.0058
7,Rede Neural (MLP),0.7481,0.6976,0.0174,0.7968,0.0067


## 6. Otimizar modelo

> TODO (Gabriel).

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

candidatos = {
    "logistica": (
        make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED, max_iter=1000)),
        {"logisticregression__C": loguniform(0.001, 100)},
    ),
    "random_forest": (
        RandomForestClassifier(random_state=SEED),
        {
            "n_estimators": randint(100, 400),
            "max_depth": randint(4, 20),
            "min_samples_leaf": randint(1, 50),
            "max_features": ["sqrt", 0.5, None],
        },
    ),
    "mlp": (
        make_pipeline(StandardScaler(), MLPClassifier(random_state=SEED, max_iter=500, early_stopping=True)),
        {
            "mlpclassifier__hidden_layer_sizes": [(16,), (32,), (64,), (32, 16)],
            "mlpclassifier__alpha": loguniform(1e-5, 1e-1),
            "mlpclassifier__learning_rate_init": loguniform(1e-4, 1e-2),
        },
    ),
}

melhores = {}
resultados = []
for nome, (modelo, parametros) in candidatos.items():
    busca = RandomizedSearchCV(modelo, parametros, n_iter=20, cv=5, scoring="roc_auc", random_state=SEED, n_jobs=4)
    busca.fit(x_train_clean, y_train_clean)
    melhores[nome] = busca.best_estimator_
    i = busca.best_index_
    resultados.append({
        "Modelo": nome,
        "ROC AUC": busca.best_score_,
        "ROC AUC (desvio)": busca.cv_results_["std_test_score"][i],
        "Melhores parâmetros": busca.best_params_,
    })

df_resultados = pd.DataFrame(resultados).round(4)


,Modelo,ROC AUC,ROC AUC (desvio),Melhores parâmetros
0,logistica,0.7918,0.0077,{'logisticregression__C': 70.72114131472235}
1,random_forest,0.8005,0.0075,"{'max_depth': 10, 'max_features': 'sqrt', 'min..."
2,mlp,0.7992,0.0078,"{'mlpclassifier__alpha': 0.07852755494724252, ..."


In [114]:
df_resultados = pd.DataFrame(resultados).round(4)
df_resultados[df_resultados["Modelo"]=="random_forest"]["Melhores parâmetros"].values


array([{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 15, 'n_estimators': 206}],
      dtype=object)

## 7. Organizar apresentação da solução

> TODO (Gabriel).

In [123]:
import shap
from sklearn.metrics import confusion_matrix

modelo_final = RandomForestClassifier(max_depth=10, max_features='sqrt', min_samples_leaf=15, n_estimators=206)
modelo_final.fit(x_train_clean, y_train_clean)

proba_teste = modelo_final.predict_proba(x_test_clean)[:, 1]
pred_teste = modelo_final.predict(x_test_clean)

In [124]:
display(df_resultados.round(4))

display(pd.DataFrame(
    confusion_matrix(y_test_clean, pred_teste),
    index=["Real: saudável", "Real: doente"],
    columns=["Previsto: saudável", "Previsto: doente"],
))

,Modelo,ROC AUC,ROC AUC (desvio),Melhores parâmetros
0,logistica,0.7918,0.0077,{'logisticregression__C': 70.72114131472235}
1,random_forest,0.8005,0.0075,"{'max_depth': 10, 'max_features': 'sqrt', 'min..."
2,mlp,0.7992,0.0078,"{'mlpclassifier__alpha': 0.07852755494724252, ..."


,Previsto: saudável,Previsto: doente
Real: saudável,8224,2253
Real: doente,3182,7283
